# Moran Process on Graph Topologies: Scaling Analysis

How fixation probability and fixation time scale with graph size, for respiratory topologies
measured against random graphs matched on both size and edge count.

Reads finished batches only. Every heavy artefact is built by a cluster job; this kernel does
file reads and plotting, nothing else.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2
%cd /home/labs/pilpel/matanyaw/moran-process

import sys

sys.path.insert(0, "src")

import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
from moran_process.analysis.analysis_utils import *
from moran_process.analysis.analysis_utils.plotly_prototype import (
    plot_outcome_vs_property_plotly,
    plot_two_property_effect_plotly,
)
from moran_process.core.population_graph import GRAPH_PROPS

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.2)
plt.rcParams["figure.figsize"] = (5.5, 4.5)
plt.rcParams["lines.linewidth"] = 2.5

## 2. Configuration

The only cell to edit per analysis.

Listing several batches stitches them at read time and adds a `batch` column, so facet or
color by it or a shared graph is drawn twice. A batch whose aggregation has not landed yet is
dropped here and reported, so `BATCH_NAMES` can list the whole study permanently: each batch
joins the figures the moment its `graph_statistics.csv` appears, with no edit.

In [ ]:
BATCH_NAMES = [
    "2026_08_31-scaling-lung-2",
    # "2026_08_31-scaling-avian",
    # "2026_08_31-scaling-gills",
    # "2026_09_02-scaling-more-shapes",
    # "2026_09_02-test-birds-proportions",
    # "2026_09_07-avian-rods-vs-length",
    # "2026_09_08-check-star-non-monotonues-behavior",
]

R_VALUE_FILTER = [1.1]
CATEGORY_FILTER = None  # None = all; else e.g. ["Random"] + ANIMAL_CATEGORIES
ANIMAL_CATEGORIES = ["Mammalian", "Fish", "Avian"]

# Must match the violin-cache job's cap (pipeline/cache_violin_data.py, default 50_000):
# it is part of the cache key, and a mismatch raises.
MAX_POINTS_PER_CATEGORY = 50_000

# Use the analytic star instead of simulating it. TODO: verify before trusting.
ADD_ANALYTIC_STAR = False

ROOT = Path(os.getcwd())
BATCH_DIRS = [ROOT / "simulation_data" / name for name in BATCH_NAMES]
FIGURES_DIR = ROOT / "figures" / "respiratory_vs_extreme"  # set per run, not derived

# Categories missing from CATEGORY_COLOR_DICT fall back to the plot helpers' auto-colors.
category_color_dict = CATEGORY_COLOR_DICT

READY_BATCH_DIRS = [d for d in BATCH_DIRS if (d / "graph_statistics.csv").exists()]
for batch_dir in BATCH_DIRS:
    if batch_dir not in READY_BATCH_DIRS:
        print(f"NOT AGGREGATED YET, excluded from every figure below: {batch_dir.name}")
BATCH_LABEL = " + ".join(d.name for d in READY_BATCH_DIRS)

## 3. What ran, and is it ready to read

Each batch is checked on its own: a stitch is only as ready as its least ready member.

In [ ]:
from moran_process.pipeline.post_batch import post_batch_status, print_post_batch_status
from moran_process.analysis.batch_speed_report import batch_speed_report
from moran_process.pipeline.job_speed import job_speed_path

for batch_dir in BATCH_DIRS:
    print_post_batch_status(post_batch_status(batch_dir, r_values=R_VALUE_FILTER))

# Fill the missing post-batch steps (force=True rebuilds). Returns at once; jobs run on LSF.
# from moran_process.pipeline.post_batch import ensure_post_batch
# ensure_post_batch(BATCH_DIRS[0])

for batch_dir, batch_name in zip(BATCH_DIRS, BATCH_NAMES):
    speed_path = job_speed_path(batch_dir)
    if not Path(speed_path).exists():
        print(f"{batch_name}: no job_speed.csv (run the post_batch chain on it). Skipped.")
        continue
    batch_speed_report(batch_name, pd.read_csv(speed_path), batch_dir)

### Optional inspections

Neither is needed by anything below: no figure reads the raw rows, they read
`graph_statistics.csv` and the violin cache, both built by jobs. Uncomment to run.

In [ ]:
# --- Batch info cards -------------------------------------------------------------------
# Register a new batch with create_batch_info(<batch_dir>, name=..., description=..., ...).
# save is off when stitching: every card resolves to the same filename.
#
# batch_infos = [load_batch_info(batch_dir) for batch_dir in BATCH_DIRS]
# for info in batch_infos:
#     plot_batch_info_card(info, figures_dir=FIGURES_DIR, save=len(BATCH_DIRS) == 1)

# --- Raw shard inventory ----------------------------------------------------------------
# Row counts come from the parquet footers, not pl.len() (u32, overflows past 4.3e9 rows).
#
# import glob
# import pyarrow.parquet as pq
#
# total = 0
# for batch_dir in BATCH_DIRS:
#     source = resolve_results_source(batch_dir)
#     files = glob.glob(str(source))
#     schema = scan_results(source).collect_schema()
#     n_rows = sum(pq.ParquetFile(f).metadata.num_rows for f in files)
#     total += n_rows
#     print(f"{batch_dir.name}: {len(files)} shard(s), {n_rows:,} rows x {len(schema)} columns")
#
# print("\ncolumns:", list(schema.keys()))
# print(f"Total across {len(BATCH_DIRS)} batch(es): {total:,} rows")

## 4. Load the aggregated statistics

One row per `(graph, r)`. Watch the load-time report for **ragged columns** (respiratory-only
construction params, NaN for GA graphs) and **`(wl_hash, r)` collisions** (the same graph in
two batches, kept as separate rows and labelled by `batch`).

In [ ]:
# Only used for the default category ordering in the violin / p-value figures.
df_graph_props = pd.concat(
    [pd.read_csv(batch_dir / "graph_props.csv") for batch_dir in READY_BATCH_DIRS],
    ignore_index=True,
    sort=False,
)

analysis_df = load_graph_statistics(
    READY_BATCH_DIRS,
    category_filter=CATEGORY_FILTER,
    r_filter=R_VALUE_FILTER,
)
# Do NOT narrow on is_directed here: the random cloud is undirected in every batch of this
# study, so it would go with them and the scaling figure would lose its baseline. Use
# DIRECTEDNESS_SPLIT below, which splits the FIGURE instead.

if ADD_ANALYTIC_STAR:
    analysis_df = add_analytic_star_rows(analysis_df)

analysis_df.tail(5)

## 5. Scaling: each organ against a matched random cloud

Each organ is one graph per size, so a scaling curve needs the 500-graph random cloud
summarized rather than scattered. The cloud is reduced to one row per size, reported as the
**median with its inter-quartile band**: the question is whether the organ falls *outside*
the cloud, which needs the cloud's spread, and the SEM of 500 graphs would draw it as a point
however wide it really is.

### Grouping keys

`(batch, category, r, is_directed, n_nodes, n_edges)`. Nothing distinct should be medianed
together silently, and each key earns its place:

- **`batch`** is a grouping key, not a label. Every plan ships its own random cloud,
  edge-matched to the organ it accompanies: at `n_nodes = 15` the lung batch's random graphs
  are trees (14 edges), the avian batch's carry 20, and the gill batch is on a different size
  ladder entirely. Pooling them would measure every curve against a baseline that is part one
  family and part another.
- **`r`** is held to a single value by a guard below, since this figure draws one point per
  `(type, size)` and has no axis left for a second `r`. Keeping it in the key is the belt to
  that guard's brace.
- **`is_directed`** is grouped rather than filtered. A single-source topology abolishes
  selection (`rho = 1/N` at every `r`), so its probability curve should lie on the neutral
  reference at every size: a live check on the pipeline, not just a null result.
- **`n_edges`** matters for a batch that varies *shape* at fixed size.
  `2026_09_07-avian-rods-vs-length` is ~20 avian graphs and ~20 random clouds all at
  `n_nodes = 1023`, separated only by their edge counts (1025..1042). Without it every avian
  graph medians into one row and the exact edge test below decides on a median of 1025..1042.

### Censoring

A run that hits `max_steps` is written as `(fixation=False, steps=max_steps)`,
indistinguishable from an extinction, so it pulls `prob_fixation` down and drops the slowest
fixations out of `mean_steps`: **both headline statistics are biased low wherever
`n_censored > 0`**. Those sizes are drawn hollow and dotted rather than deleted, so the damage
stays on the figure instead of silently flattening a trend. All three scaling batches ran at
`max_steps = 1e8` and currently report zero censoring, so the guard should stay quiet; it is
kept because it is the only thing that would tell you if that changed.

### Which types get drawn

A random cloud controls for size and edge budget, or it controls for nothing, so a type is
drawn only when both line up. Everything else is dropped with its reason printed, and appears
in the overview below, which claims no control. All three rejections are live here: a
shape-at-fixed-size batch has no single point per size, the grid sits at perfect rectangles
while every cloud sits at `2^k - 1`, and the complete graph carries 63x the edges.

Two fallbacks are annotated onto the figure rather than assumed away:

- a cloud **borrowed** from another batch, for a batch whose plan has no random spec at all
  (`2026_09_02-scaling-more-shapes` is 7 shapes x 7 sizes and nothing else);
- an **undirected** cloud standing in for a directed type, which happens for every directed
  type here since no plan draws directed random graphs. `n_edges` then agrees only in the
  networkx counting sense: 14 arcs and 14 undirected edges are 14 and 28 movement directions.

### The three panels

Fixation time, fixation probability, and their **ratio**. The ratio panel is where the
scaling signal lives and the only place the cloud's spread in time is legible: panel 1 draws
the same IQR, but on a log axis spanning 3.5 decades a +/-7% band is about one line-width
tall. Dividing the shared ~N^2 growth out lets it be drawn at full size. The band is the
random cloud's IQR over the organ's single value, because the organ is one graph per size and
has no spread of its own (its Monte Carlo SEM is under 0.5% of the mean, against a cloud IQR
of 3.7% to 27%).

In [ ]:
# Highest pooled censored fraction a size may carry and still count as trustworthy.
CENSOR_TOLERANCE = 0.005

# Relative tolerance on the organ-vs-cloud edge count; 0 means they must be equal.
EDGE_TOLERANCE = 0.0

# One FIGURE per entry, so directed and undirected land on separate slides. This selects
# which TYPES get drawn and never touches the cloud (see the note in section 4).
DIRECTEDNESS_SPLIT = [False, True]

r_values_present = sorted(analysis_df["r"].unique())
if len(r_values_present) > 1:
    raise ValueError(
        f"the scaling figure draws one r at a time, found {r_values_present}. "
        f"Narrow R_VALUE_FILTER in the configuration cell."
    )

GROUP_KEYS = ["batch", "category", "r", "is_directed", "n_nodes", "n_edges"]

scaling_df = (
    analysis_df.groupby(GROUP_KEYS)
    .agg(
        n_graphs=("wl_hash", "size"),
        edges=("n_edges", "median"),  # half of the match test below
        prob=("prob_fixation", "median"),
        prob_lo=("prob_fixation", lambda s: s.quantile(0.25)),
        prob_hi=("prob_fixation", lambda s: s.quantile(0.75)),
        steps=("mean_steps", "median"),
        steps_lo=("mean_steps", lambda s: s.quantile(0.25)),
        steps_hi=("mean_steps", lambda s: s.quantile(0.75)),
        neutral_prob=("fc_prob_fixation", "median"),  # well-mixed Moran rho, for reference
        n_censored=("n_censored", "sum"),
        n_runs=("n_grouped", "sum"),
    )
    .reset_index()
    .sort_values(GROUP_KEYS)
)
# 0/0 for an exact row (the analytic star runs no simulations): no runs means nothing could
# have hit max_steps, so that is a censored fraction of zero, not an unknown one.
scaling_df["censored_frac"] = (scaling_df["n_censored"] / scaling_df["n_runs"]).fillna(0.0)
scaling_df["trusted"] = scaling_df["censored_frac"] <= CENSOR_TOLERANCE

# A non-random row is meant to be ONE graph, since that is what "the mammalian lung at N = 31"
# means. Two distinct topologies sharing a full key is a legitimate zoo, but then the point
# drawn is no single graph, so say so rather than let the median pass as a measurement.
multi = scaling_df[(scaling_df["category"] != "Random") & (scaling_df["n_graphs"] > 1)]
if len(multi):
    print("WARNING: these rows median over several DISTINCT topologies, so the point "
          "drawn below is not any one graph:")
    print(multi[GROUP_KEYS + ["n_graphs"]].to_string(index=False), "\n")


def rows_for(batch, category, is_directed):
    """The scaling curve of one graph type in one batch."""
    return scaling_df[
        (scaling_df["batch"] == batch)
        & (scaling_df["category"] == category)
        & (scaling_df["is_directed"] == is_directed)
    ]


def cloud_for(batch, is_directed, sizes):
    """The random cloud a given variant is tested against.

    Own batch and directedness first, else that batch's undirected cloud, else the ONE other
    batch's cloud covering the most of `sizes` (ties broken by name, so the pick is
    deterministic). Never a POOL of batches: each plan's cloud is edge-matched to its own
    organ, so medianing two would blend two densities and describe no batch.
    """

    def pick(df):
        same = df[df["is_directed"] == is_directed]
        return same if len(same) else df[~df["is_directed"]]

    random_rows = scaling_df[scaling_df["category"] == "Random"]
    own = pick(random_rows[random_rows["batch"] == batch])
    if len(own):
        return own
    borrowed = [pick(g) for _, g in random_rows.groupby("batch", sort=True)]
    borrowed = [c for c in borrowed if len(c)]
    if not borrowed:
        return random_rows.iloc[:0]  # no cloud in any loaded batch; the row is skipped
    return max(borrowed, key=lambda c: len(set(sizes) & set(c["n_nodes"])))


def paired_or_reason(rows, cloud):
    """(rows joined to the cloud, None) when the cloud is a valid control, else (None, why).

    An INNER join on n_nodes, so nothing is imputed and no size is paired with an approximate
    neighbour. Edge budgets must agree too: a cloud of a different density answers a question
    about density rather than about shape.
    """
    if rows["n_nodes"].duplicated().any() or cloud["n_nodes"].duplicated().any():
        return None, ("several rows share one n_nodes, so there is no single point per size "
                      "to draw. That is a batch varying SHAPE at fixed size rather than a "
                      "scaling batch; read it in the overview below instead")
    shared = rows.merge(cloud, on="n_nodes", suffixes=("", "_rnd"))
    if len(shared) < len(rows):
        missing = sorted(set(rows["n_nodes"]) - set(cloud["n_nodes"]))
        return None, (f"n_nodes {missing} are not in the cloud, which sits at "
                      f"{sorted(cloud['n_nodes'])}")
    gap = (shared["edges"] - shared["edges_rnd"]).abs()
    off = shared[gap > EDGE_TOLERANCE * shared["edges_rnd"]]
    if len(off):
        w = off.iloc[-1]
        return None, (f"n_edges differ at {len(off)} of {len(shared)} sizes, "
                      f"e.g. N={int(w['n_nodes'])}: {int(w['edges'])} vs {int(w['edges_rnd'])}")
    return shared, None


def cloud_note(batch, cloud, is_directed):
    """Every way this cloud falls short of a like-for-like control. Empty when it does not."""
    notes = []
    src = cloud["batch"].iloc[0]
    if src != batch:
        notes.append(f"cloud borrowed from {src}")
    cloud_directed = bool(cloud["is_directed"].iloc[0])
    if cloud_directed != is_directed:
        kinds = ("undirected", "directed")
        notes.append(f"cloud is {kinds[cloud_directed]} and this type is "
                     f"{kinds[is_directed]}: n_edges matches, movement directions do not")
    return "\n".join(notes)


def cloud_size_label(cloud):
    """Graphs per size, as a range when dedup left the cloud ragged rather than as a max."""
    lo, hi = cloud["n_graphs"].min(), cloud["n_graphs"].max()
    return f"{lo}" if lo == hi else f"{lo}-{hi}"


# One figure row per graph type that the cloud can actually control for.
TYPE_ROWS, dropped = [], []
for batch, category, is_directed in (
    scaling_df.loc[scaling_df["category"] != "Random", ["batch", "category", "is_directed"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
):
    is_directed = bool(is_directed)
    rows = rows_for(batch, category, is_directed)
    cloud = cloud_for(batch, is_directed, rows["n_nodes"])
    if cloud.empty:
        dropped.append((category, is_directed, batch, "no Random rows in any loaded batch"))
        continue
    paired, reason = paired_or_reason(rows, cloud)
    if reason:
        dropped.append((category, is_directed, batch, reason))
        continue
    TYPE_ROWS.append((batch, category, is_directed, cloud, paired))

for category, is_directed, batch, reason in dropped:
    kind = "directed" if is_directed else "undirected"
    print(f"  no matched cloud, not drawn: {category} ({kind}) [{batch}]\n"
          f"      {reason}")

print(f"\n{len(TYPE_ROWS)} graph type(s) matched to a random cloud on n_nodes AND n_edges:")
for batch, category, is_directed, cloud, _ in TYPE_ROWS:
    kind = "directed" if is_directed else "undirected"
    note = cloud_note(batch, cloud, is_directed)
    print(f"  {category:<10} {kind:<11} vs {cloud_size_label(cloud)} random per size, "
          f"n_nodes {cloud['n_nodes'].tolist()}   [{batch}]"
          + (f"\n{'':<26}^ {note}" if note else ""))

scaling_df.drop(columns=["n_censored", "n_runs"]).head()

In [ ]:
CLOUD_COLOR = "#9E9E9E"  # CATEGORY_COLOR_DICT["Random"] is near-white: invisible as a line


def draw_scaling(ax, sub, y, color, label=None, lo=None, hi=None, ms=8):
    """One scaling curve: solid while trustworthy, dotted and hollow once censored."""
    sub = sub.sort_values("n_nodes").reset_index(drop=True)  # positional slicing below
    if lo is not None:
        ax.fill_between(sub["n_nodes"], sub[lo], sub[hi], color=color, alpha=0.3, lw=0)
    cut = (~sub["trusted"]).values.argmax() if not sub["trusted"].all() else len(sub)
    tail = max(cut - 1, 0)  # cut == 0 would make iloc[-1:] wrap to the last row
    ax.plot(sub["n_nodes"].iloc[:cut], sub[y].iloc[:cut], "-", color=color, label=label,
            zorder=3)
    ax.plot(sub["n_nodes"].iloc[tail:], sub[y].iloc[tail:], ":", color=color, zorder=3)
    ok, bad = sub[sub["trusted"]], sub[~sub["trusted"]]
    ax.plot(ok["n_nodes"], ok[y], "o", color=color, ms=ms, zorder=4)
    ax.plot(bad["n_nodes"], bad[y], "o", mfc="white", mec=color, ms=ms, mew=2.5, zorder=4)


def draw_type_row(batch, category, is_directed, cloud, paired, axes_row):
    """The three panels for one graph type: fixation time, probability, speed-up."""
    ax_time, ax_prob, ax_ratio = axes_row
    animal = rows_for(batch, category, is_directed)
    color = CATEGORY_COLOR_DICT.get(category, "black")
    kind = "directed" if is_directed else "undirected"
    animal_label = f"{category} ({kind})"
    cloud_kind = "directed" if bool(cloud["is_directed"].iloc[0]) else "undirected"
    cloud_label = f"Random {cloud_kind} (n={cloud_size_label(cloud)} per size)"

    draw_scaling(ax_time, cloud, "steps", CLOUD_COLOR, cloud_label, "steps_lo", "steps_hi")
    draw_scaling(ax_time, animal, "steps", color, animal_label)
    draw_scaling(ax_prob, cloud, "prob", CLOUD_COLOR, cloud_label, "prob_lo", "prob_hi")
    draw_scaling(ax_prob, animal, "prob", color, animal_label)
    ax_prob.plot(cloud["n_nodes"], cloud["neutral_prob"], "--", color="black", lw=1.2,
                 label=r"well-mixed Moran $\rho$")

    ax_time.set(xscale="log", yscale="log", xlabel="n_nodes",
                ylabel="mean steps to fixation",
                title=f"{animal_label}: fixation time")
    ax_prob.set(xscale="log", xlabel="n_nodes", ylabel=r"$\rho$",
                title="fixation probability")
    # Provenance of the COMPARISON, not of the data: a borrowed cloud, or one of the other
    # directedness, is exactly what a reader would otherwise assume away.
    note = cloud_note(batch, cloud, is_directed)
    ax_time.annotate(batch + (f"\n{note}" if note else ""), (0.02, 0.96),
                     xycoords="axes fraction", va="top", fontsize=8, color="gray")

    # `paired` is the size-for-size join the cell above already validated.
    ratio = pd.DataFrame({
        "n_nodes": paired["n_nodes"],
        "ratio": paired["steps_rnd"] / paired["steps"],
        "ratio_lo": paired["steps_lo_rnd"] / paired["steps"],
        "ratio_hi": paired["steps_hi_rnd"] / paired["steps"],
        "trusted": paired["trusted"] & paired["trusted_rnd"],
    }).dropna(subset=["ratio"])
    ax_ratio.axhline(1.0, color="black", ls="--", lw=1.2)
    draw_scaling(ax_ratio, ratio, "ratio", color, lo="ratio_lo", hi="ratio_hi")
    ax_ratio.set(xscale="log", xlabel="n_nodes",
                 ylabel=f"random / {category.lower()}  (mean steps)",
                 title="speed-up over random (band = random IQR)")
    # Only tag the "organ slower" half when the curve reaches it, else the tag lands on the
    # axhline it is meant to explain.
    if ratio["ratio"].min() < 1:
        ax_ratio.annotate(f"{category.lower()} slower", (0.02, 0.03),
                          xycoords="axes fraction", fontsize=10, color="gray")

    ax_time.legend(fontsize=9)
    ax_prob.legend(fontsize=9)


r_label = ", ".join(f"{v:g}" for v in sorted(analysis_df["r"].unique()))

for want_directed in DIRECTEDNESS_SPLIT:
    rows_here = [t for t in TYPE_ROWS if t[2] == want_directed]
    kind = "directed" if want_directed else "undirected"
    if not rows_here:
        print(f"no {kind} graph type has a matched cloud, so no {kind} figure is drawn")
        continue
    fig, axes = plt.subplots(len(rows_here), 3, figsize=(17, 4.6 * len(rows_here)),
                             squeeze=False)
    for type_row, axes_row in zip(rows_here, axes):
        draw_type_row(*type_row, axes_row)
    fig.suptitle(f"Each {kind} graph type against a random cloud matched on n_nodes and "
                 f"n_edges, r = {r_label}", y=1 - 0.12 / fig.get_figheight())
    fig.tight_layout(rect=(0, 0, 1, 1 - 0.5 / fig.get_figheight()))
    plt.show()

### 5b. Every category on one axis

The figure above draws a type only when the random cloud is a genuine control. That leaves
the complete graph and the grid out. They are still worth seeing, just not against a cloud
that answers a different question, so they go here: all categories, **no control claimed**,
hover for the exact numbers. Random is its per-size median, one point per size rather than
500, so it reads as a curve alongside the single-graph types.

In [ ]:
import plotly.express as px

# CATEGORY_COLOR_DICT["Random"] is near-white, which vanishes on a white plotly background.
OVERVIEW_COLORS = {**CATEGORY_COLOR_DICT, "Random": CLOUD_COLOR}

overview_df = scaling_df.sort_values(["category", "is_directed", "n_nodes"])

for metric, axis_label, log_y in (
    ("steps", "median mean steps to fixation", True),
    ("prob", "median fixation probability", False),
):
    fig = px.line(
        overview_df,
        x="n_nodes",
        y=metric,
        color="category",
        line_dash="is_directed",
        markers=True,
        color_discrete_map=OVERVIEW_COLORS,
        log_x=True,
        log_y=log_y,
        hover_data=["batch", "n_graphs", "edges", "trusted"],
        labels={"n_nodes": "N (nodes)", metric: axis_label, "is_directed": "directed"},
        title=f"{axis_label} vs size, every category, r = {r_label}",
    )
    if metric == "prob":
        # This is the only figure covering Complete and Grid, which never reach the matched
        # figure above, so without this line they would have no analytic baseline anywhere.
        # r is single-valued, so neutral_prob is a function of n_nodes alone.
        ref = (overview_df[["n_nodes", "neutral_prob"]]
               .drop_duplicates().sort_values("n_nodes"))
        fig.add_scatter(x=ref["n_nodes"], y=ref["neutral_prob"], mode="lines",
                        line=dict(color="black", dash="dash", width=1.5),
                        name="well-mixed Moran rho")
    fig.update_layout(height=520, legend_title_text="", hovermode="closest")
    fig.show()

## 6. The joint (probability, time) plane

Where each topology sits on both headline metrics at once. An amplifier should show elevated
probability relative to neutral drift (`rho = 1/N`); read it together with the time axis.

In [ ]:
plot_outcome_vs_property_plotly(
    analysis_df,
    "mean_steps",
    "prob_fixation",
    density_threshold=None,
    color_dict=CATEGORY_COLOR_DICT,
    filter_categories=CATEGORY_FILTER,
)

In [ ]:
plot_outcome_vs_property(
    analysis_df,
    "mean_steps",
    "prob_fixation",
    density_threshold=None,
    color_dict=category_color_dict,
    figures_dir=FIGURES_DIR,
    batch_name=BATCH_LABEL,
    save=False,
    filter_categories=CATEGORY_FILTER,
    corr=None,
)

In [ ]:
plot_outcome_vs_property_plotly(
    analysis_df,
    "std_steps",
    "mean_steps",
    color_dict=category_color_dict,
    highlight_categories=ANIMAL_CATEGORIES,
    # corr="spearman",
)

## 7. Structural property sweeps

One structural property per plot, against fixation time then fixation probability. Animal
graphs are highlighted; random graphs form the background cloud. Look for properties where
the animal graphs sit outside that cloud.

In [ ]:
for prop, desc in GRAPH_PROPERTY_DESCRIPTION.items():
    print(f"{prop.title().replace('_', ' ')}: {desc}")

### Fixation time

In [ ]:
for prop in GRAPH_PROPS:
    plot_outcome_vs_property_plotly(
        analysis_df,
        prop,
        "mean_steps",
        density_threshold=50,
        color_dict=category_color_dict,
        highlight_categories=ANIMAL_CATEGORIES,
    ).show()

### Fixation probability

In [ ]:
for prop in GRAPH_PROPS:
    plot_outcome_vs_property(
        analysis_df,
        prop,
        "prob_fixation",
        density_threshold=50,
        color_dict=category_color_dict,
        highlight_categories=ANIMAL_CATEGORIES,
        figures_dir=FIGURES_DIR,
        batch_name=BATCH_LABEL,
        save=False,
    )

## 8. Avian shape sweep: n_rods against rod_length

One batch only: a full 32 x 32 grid of `(n_rods, rod_length)` at both directednesses. The
cell reads the avian construction parameters and **raises** on a batch that has none, which
is every scaling batch above. That is deliberate, it is meant to be skipped rather than
generalised.

- **Top row**: the two headline metrics against `n_rods`, one line per `rod_length`.
- **Bottom row** divides graph size back out. `avian_graph` builds
  `n_nodes = n_rods * rod_length + 3`, so every step to the right also makes the graph bigger:
  measured on this batch at `rod_length = 4`, `mean_steps` moves 2104x while `N` moves 19x, so
  the top row is mostly a size curve wearing a shape label. The bottom row plots each graph
  against the analytic complete graph of the **same N**, which is what
  `log_ratio_mean_steps` and `delta_prob_fixation` already hold. Zero means "behaves like a
  well-mixed population of its own size".

In [ ]:
# Batch-specific ON PURPOSE: raises rather than silently drawing an empty figure.
SWEEP_BATCH = "2026_09_02-test-birds-proportions"
SHOW_SIZE_CONTROLLED = True  # the bottom row; see the markdown above for why it is on

sweep = analysis_df[analysis_df["batch"] == SWEEP_BATCH]
if sweep.empty:
    raise ValueError(
        f"{SWEEP_BATCH} is not loaded. Add it to BATCH_NAMES in the configuration cell."
    )
# avian_graph records its rod length under 'rods_length' and fish_graph under 'rod_length',
# and neither matches the factory kwarg (rod_length), so it is resolved rather than assumed.
LEN_COL = next((c for c in ("rods_length", "rod_length")
                if c in sweep.columns and sweep[c].notna().any()), None)
if LEN_COL is None or sweep["n_rods"].isna().all():
    raise ValueError(f"{SWEEP_BATCH} carries no rod parameters; this figure is avian only.")
sweep = sweep.dropna(subset=["n_rods", LEN_COL])

# Censoring biases BOTH metrics low, so say it rather than let a flattened curve pass.
n_bad = int((sweep["n_censored"] > 0).sum()) if "n_censored" in sweep.columns else 0
if n_bad:
    print(f"WARNING: {n_bad} of {len(sweep)} graphs hit max_steps; raise --max-steps.")

PANELS = [("mean_steps", "mean steps to fixation", True),
          ("prob_fixation", r"fixation probability $\rho$", False)]
if SHOW_SIZE_CONTROLLED:
    PANELS += [("log_ratio_mean_steps", r"$\log(T\,/\,T_{\rm complete})$, same $N$", False),
               ("delta_prob_fixation", r"$\rho - \rho_{\rm complete}$, same $N$", False)]

# rod_length is ordinal with 32 levels, so it gets a colorbar rather than a 32-row legend.
norm = plt.Normalize(sweep[LEN_COL].min(), sweep[LEN_COL].max())
cmap = plt.get_cmap("viridis")

for want_directed in DIRECTEDNESS_SPLIT:
    sub = sweep[sweep["is_directed"] == want_directed]
    kind = "directed" if want_directed else "undirected"
    if sub.empty:
        print(f"no {kind} graphs in {SWEEP_BATCH}, no figure drawn")
        continue
    nrows = (len(PANELS) + 1) // 2
    fig, axes = plt.subplots(nrows, 2, figsize=(13, 4.6 * nrows), squeeze=False,
                             constrained_layout=True)
    for (col, label, log_y), ax in zip(PANELS, axes.flat):
        for rod_len, grp in sub.groupby(LEN_COL):
            grp = grp.sort_values("n_rods")
            ax.plot(grp["n_rods"], grp[col], "-", lw=1.3, color=cmap(norm(rod_len)))
        if col.startswith(("log_ratio", "delta_")):
            ax.axhline(0.0, color="black", ls="--", lw=1.2)  # the complete graph itself
        ax.set(xscale="log", xlabel="n_rods", ylabel=label)
        if log_y:
            ax.set_yscale("log")
    for ax in axes.flat[len(PANELS):]:
        ax.set_visible(False)
    fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=axes,
                 label="rod_length", fraction=0.03, pad=0.02)
    fig.suptitle(f"Avian {kind}: both metrics vs n_rods, one line per rod_length "
                 f"(N = n_rods x rod_length + 3, {int(sub['n_nodes'].min())} to "
                 f"{int(sub['n_nodes'].max())}), r = {r_label}   [{SWEEP_BATCH}]")
    plt.show()

## 9. Scratch

Ad-hoc views kept for convenience. Nothing below is part of the standard read-through.

In [ ]:
plot_two_property_effect(
    analysis_df,
    "n_nodes",
    "n_edges",
    "mean_steps",
    color_dict=category_color_dict,
    save=False,
    descriptions_below=True,
)

In [ ]:
analysis_df.columns

In [ ]:
plot_two_property_effect(
    analysis_df,
    x_prop="n_rods",
    y_prop="rods_length",
    outcome="mean_steps",
    save=False
    
      )

In [ ]:
plot_outcome_vs_property(
    analysis_df[analysis_df["is_directed"] == False],
    "n_rods",
    "mean_steps",
    density_threshold=50,
    color_dict=category_color_dict,
    highlight_categories=ANIMAL_CATEGORIES,
    figures_dir=FIGURES_DIR,
    batch_name=BATCH_LABEL,
    save=False,
)

In [ ]:
plot_outcome_vs_property(
    analysis_df,
    "n_nodes",
    "mean_steps",
    density_threshold=50,
    color_dict=category_color_dict,
    highlight_categories=ANIMAL_CATEGORIES,
    figures_dir=FIGURES_DIR,
    batch_name=BATCH_LABEL,
    save=False,
)

In [ ]:
plot_outcome_vs_property(
    analysis_df,
    "degree_std",
    "mean_steps",
    density_threshold=50,
    color_dict=category_color_dict,
    batch_name=BATCH_LABEL,
    save=False,
)

plt.figure(figsize=(10, 8))
plt.hexbin(
    analysis_df["degree_std"],
    analysis_df["mean_steps"],
    gridsize=20,
    cmap="YlOrRd",
    mincnt=1,
)
plt.xlabel("degree_std")
plt.ylabel("mean_steps")
plt.colorbar(label="count")
plt.title("Hexbin plot: degree_std vs mean_steps")
plt.show()

In [ ]:
# plot_steps_histogram(analysis_df, metric='mean_steps', figures_dir=FIGURES_DIR,
#                      batch_name=BATCH_LABEL, save=False)
# plot_steps_histogram(analysis_df, metric='mean_steps', category='Random',
#                      figures_dir=FIGURES_DIR, batch_name=BATCH_LABEL, save=False)